# Data Download and Analysis

#### In this notebook I will populate the `stations` and `trips` tables by connecting to the citibike database and uploading the data

### Imports

In [ ]:
import sys
sys.executable
sys.path
!{sys.executable} -m pip install psycopg2-binary

In [ ]:
import pandas as pd
import json
import os
import psycopg2
from psycopg2.extras import execute_values
import requests

from pathlib import Path

## Upload data to stations table

### Download the  data from CitiBike API

To get data from lightsail run in terminal
scp -i /path/to/your-key.pem <instance-username>@<instance-public-ip>:/path/on/instance/file.txt /path/on/local/machine/


In [ ]:
url = "https://gbfs.lyft.com/gbfs/2.3/bkn/en/station_information.json"

response =requests.get(url)

In [ ]:
#print the keys in the json file
print(f'Keys in the station info json file: {response.json().keys()}')


In [ ]:
#extract stations from response
stations = response.json()['data']['stations']
#inspect data
print('The number of stations in data',len(stations))
stations[:3]

In [ ]:
#Make a dataframe for stations
df = pd.DataFrame(stations)
df.head()

In [ ]:
df.shape

In [ ]:
df['region_id'].unique()

In [ ]:
#dataframe columns 
print('Dataframe columns',df.columns.tolist())

The stations table in the database has the following columns:
- station_id
- name
- short_name
- latitude
- longitude
- capacity
So only include these columns in the dataframe

### - Include only relevant columns in the dataframe and call it df_clean and clean nulls

In [ ]:
df_clean = df[['station_id','name','short_name','lat','lon','capacity']].copy()
df_clean.head(2)

In [ ]:
# Clean df from nans
print(f'nulls in data: \n{df_clean.isnull().sum()}')

### - Connect to database citibike

In [ ]:
DB_HOST=os.environ.get('DB_HOST')
DB_NAME=os.environ.get('DB_NAME')
DB_USER=os.environ.get('DB_USER')
DB_PASSW=os.environ.get('DB_PASSWORD')
DB_PORT=os.environ.get('DB_PORT')

In [ ]:
#Connect to database citikike
try:
    conn = psycopg2.connect(
            host = DB_HOST,
            database = DB_NAME,
            user = DB_USER,
            password = DB_PASSW,
            port = DB_PORT,
            keepalives=1,
            keepalives_idle=30,
            keepalives_interval=10,
            keepalives_count=5
    )
    print('Connection Successful')

    cursor = conn.cursor()
except Exception as e:
    print(f'Could not establish connection:{e}')

##  Insert data in stations table

In [ ]:
#convert the rows into tuples
tuple_records = list(df_clean.itertuples(index=False, name = None))

#insert values
try:
    execute_values(
        cursor,
        """
        INSERT INTO stations (station_id, name, short_name, latitude, longitude, capacity)
        VALUES %s
        """,
        tuple_records
    )
    conn.commit()
    print(f'Inserted {len(tuple_records)}')
except Exception as e:
    #prevents a failed partial insert from leaving the database in a messy state
    conn.rollback()
    print(f'error:{e}')

### - Check if data was uploaded

In [ ]:

cursor.execute('SELECT COUNT(*) FROM stations')
print(f'Number of records uploaded = {cursor.fetchone()}')

### Add the `is_active` column

In [ ]:
cursor.execute(
        """
        ALTER TABLE stations
        ADD COLUMN is_active BOOLEAN DEFAULT TRUE
        """
   
)
conn.commit()

Add `last_updated' column

In [ ]:
cursor.execute(
        """
        ALTER TABLE stations
        ADD COLUMN last_updated BOOLEAN DEFAULT TRUE
        """
   
)
conn.commit()

### - print one record

In [ ]:
cursor.execute('SELECT * FROM stations')
cursor.fetchone()

In [ ]:
#print columns
cursor.execute('''
SELECT COLUMN_NAME
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_NAME = 'stations';
''')
cursor.fetchall()

In [ ]:
conn.close()
print('Connection closed')


## Explore S3 CSV data for the trips table
- The december data was downloaded from citibike site, it is saved in the data folder

### Set path to CSV file

In [ ]:
cwd = Path("/Users/Rime/Documents/Eastern_University/Capstone_project_691/project_files")
data_dir = Path(cwd / 'data')
data_dir

______


# Read historic csv data downloaded from  

In [ ]:
#December data
historic_dt = pd.read_csv(data_dir / "citibike-tripdata" / "202512-citibike-tripdata_1.csv")#,nrows=10000)


In [ ]:
#January data
hist_jan_1 = pd.read_csv(data_dir / '202601-citibike-tripdata' / '202601-citibike-tripdata_1.csv')
hist_jan_1.shape

In [ ]:
hist_jan_2 = pd.read_csv(data_dir / '202601-citibike-tripdata' / '202601-citibike-tripdata_2.csv')
hist_jan_2.shape

In [ ]:
hist_jan_2.head(1)

In [ ]:
hist_jan_2.tail(2)

In [ ]:
print(f'number of columns in data = {historic_dt.shape[1]}, Number of rows = {historic_dt.shape[0]}')

In [ ]:
historic_dt.info()

In [ ]:
historic_dt.head()

In [ ]:
#columns
historic_dt.columns.tolist()

### Explore if the short_name field in the station_information data matches the start and end station ids

- Create a set for the short_names and the start/end station ids

In [ ]:
df_clean_set_ids = set(df_clean['short_name'].dropna().unique())
trips_station_ids_strt = set(historic_dt['start_station_id'].dropna().unique())
trips_station_ids_end = set(historic_dt['end_station_id'].dropna().unique())
print(f'Size of CSV start station id field = {len(trips_station_ids_strt)}, \nSize of CSV end station id field = {len(trips_station_ids_end)}, \nSize of GBFS station id field = {len(df_clean_set_ids)}')




- Calculate matches between CSV and GBFS fields

In [ ]:
matches = trips_station_ids_strt.intersection(df_clean_set_ids)
no_matches = trips_station_ids_strt - df_clean_set_ids
matches_end = trips_station_ids_end.intersection(df_clean_set_ids)

In [ ]:
#print(f"Trip station IDs: {len(trip_ids)}")
#print(f"GBFS short_names: {len(short_names)}")
print(f"Matches between start station id and GBFS station.short_name = {len(matches)}")
print(f"No match: {len(no_matches)}")
print(f'Percentage of matches = {round(len(matches)/len(df_clean_set_ids) * 100,2)}%')


# Establish a connection to the lightsail instance

In [ ]:
DB_HOST=os.environ.get('DB_HOST')
DB_NAME=os.environ.get('DB_NAME')
DB_USER=os.environ.get('DB_USER')
DB_PASSW=os.environ.get('DB_PASSWORD')
DB_PORT=os.environ.get('DB_PORT')

In [ ]:
#Connect to database citikike
try:
    conn = psycopg2.connect(
            host = DB_HOST,
            database = DB_NAME,
            user = DB_USER,
            password = DB_PASSW,
            port = DB_PORT,
            keepalives=1,
            keepalives_idle=30,
            keepalives_interval=10,
            keepalives_count=5
    )
    print('Connection Successful')

    cursor = conn.cursor()
except Exception as e:
    print(f'Could not establish connection:{e}')

## Load the S3 CSV December data into trips table

In [ ]:
#import CSV files there are three files
hist_df_1 = pd.read_csv(data_dir / "citibike-tripdata" / "202512-citibike-tripdata_1.csv")
hist_df_2 = pd.read_csv(data_dir / "citibike-tripdata" / "202512-citibike-tripdata_2.csv")
hist_df_3 = pd.read_csv(data_dir / "citibike-tripdata" / "202512-citibike-tripdata_3.csv")

print(f'Size of the csv data\nfirst file: {len(hist_df_1)}\nsecond file: {len(hist_df_2)}\nthird file: {len(hist_df_3)}\n\
for a total of = {len(hist_df_1) + len(hist_df_2)+len(hist_df_3)} rows')

- The data is added to the database in chunks. This is to improve performance, manage memory constraints and reduce network overhead during transmission

In [ ]:
def load_trips_data(filepath, conn, chunk_size=50000):
    '''
    - divideds the data in chunks of 50,000 records
    - changes the started_at and ended_at columns to datetime 
    - calculate the difference between start and end in seconds
    - uploads the columns to citibike database
     '''

    for chunk in pd.read_csv(filepath, chunksize=chunk_size,dtype={'start_station_id': str, 'end_station_id': str}):
        chunk['started_at'] = pd.to_datetime(chunk['started_at'])
        chunk['ended_at'] = pd.to_datetime(chunk['ended_at'])

        #calculate the duration
        chunk['duration_seconds'] = (chunk['ended_at'] - chunk['started_at']).dt.total_seconds().astype(int)

        #check
        print('chunk')
        print('less than zero',(chunk['duration_seconds'] < 0).sum())
        print('number of nulls',chunk['duration_seconds'].isnull().sum())


        '''
        Insert values into the database in the trips table
        It will be done in chunks so as not to overwhelm the system and
        for efficiency. Chunk size is 50,000 rows 
        '''
        cursor = conn.cursor()
        
        cols = ['ride_id','rideable_type','started_at','ended_at','start_station_id','end_station_id',
               'start_station_name','end_station_name','start_lat','start_lng','end_lat',
               'end_lng','member_casual','duration_seconds']
    
        records = list(chunk[cols].itertuples(index=False, name=None)) #tuples
    
        try:
            execute_values(
                cursor,
                
            '''
                INSERT INTO trips ( ride_id, rideable_type,started_at,ended_at,start_station_id, end_station_id,start_station_name,
                end_station_name,start_lat,start_lng,end_lat,end_lng,member_casual,duration_seconds)
                VALUES %s
                ''',
                records
            )
            #commit insert
            conn.commit()
            print(f'Inserted {len(records)} rows')
            

        except Exception as e:
            print(f'Error {e}')
            conn.rollback()
            return
    print(f'Completed {filepath}')
   



In [ ]:
#test on the first citibike file
load_trips_data(data_dir / "citibike-tripdata" / "202512-citibike-tripdata_1.csv", conn)

In [ ]:
#check how many records were uploaded, there supposed to be 1,000,000
cursor.execute('SELECT COUNT(*) FROM trips')
cursor.fetchone()

- upload the two CSV files left

In [ ]:
trip_1 = pd.read_csv(data_dir / "citibike-tripdata" / "202512-citibike-tripdata_1.csv")
trip_2 = pd.read_csv(data_dir / "citibike-tripdata" / "202512-citibike-tripdata_2.csv")
trip_3 = pd.read_csv(data_dir / "citibike-tripdata" / "202512-citibike-tripdata_3.csv")
trip_df = pd.concat([trip_1, trip_2, trip_3], axis=0)


In [ ]:
trip_df.shape

In [ ]:
filelist = [data_dir / "citibike-tripdata" / "202512-citibike-tripdata_2.csv", data_dir / "citibike-tripdata" / "202512-citibike-tripdata_3.csv"]
for file in filelist:
    load_trips_data(file,conn)

## Load January trips data
- Since there are some stations that will be added in January so I have to add them to the stations table


#### Read the stations from the stations table


In [ ]:
cursor.execute("SELECT short_name FROM stations")
existing_short_names = {row[0].strip() for row in cursor.fetchall()}

len(existing_short_names)

### Load the S3 CSV January data into trips table
- There are 2 files

In [ ]:
hist_jan_1 = pd.read_csv(data_dir / '202601-citibike-tripdata' / '202601-citibike-tripdata_1.csv')
hist_jan_2 = pd.read_csv(data_dir / '202601-citibike-tripdata' / '202601-citibike-tripdata_2.csv')

- Concat the two January files  

In [ ]:
jan_hist_df = pd.concat([hist_jan_1 ,hist_jan_2],axis=0)
#drop nas
jan_hist_df['start_station_id'] = jan_hist_df['start_station_id'].astype(str).str.strip().dropna()
#jan_hist_df = jan_hist_df.dropna(subset='start_station_id')
jan_hist_df['end_station_id'] = jan_hist_df['end_station_id'].astype(str).str.strip().dropna()
#jan_hist_df = jan_hist_df.dropna(subset='end_station_id')
jan_hist_df.shape

In [ ]:
sum(jan_hist_df['start_station_id'].astype(str).str.contains('Lab'))

- Find all the new stations that were added in January

In [ ]:
sum(jan_hist_df['end_station_id'] == '5872.1')

In [ ]:
#all station ids in the new january data
start_id = set(jan_hist_df['start_station_id'])
#start_id = set(start_id)

end_id = set(jan_hist_df['end_station_id'])
#end_id = set(end_id) 

#union start and end ids
all_ids = start_id | end_id
#all_ids = all_ids.dropna()
len(all_ids)

- Take out ids already in stations table

In [ ]:
new_stations =  all_ids - existing_short_names 
len(new_stations)

In [ ]:
new_stations

In [ ]:
new_stations.remove('Lab - NYC - Monolith')
new_stations

- Save start/end station info that needs to be added to stations

In [ ]:
start_stn_info = jan_hist_df[['start_station_id', 'start_station_name', 'start_lat', 'start_lng']].drop_duplicates(subset='start_station_id')

end_stn_info = jan_hist_df[['end_station_id', 'end_station_name', 'end_lat', 'end_lng']].drop_duplicates(subset='end_station_id')
start_stn_info.shape, end_stn_info.shape

- rename columns to match columns in the stations table

In [ ]:

start_stns = start_stn_info.rename(columns={
                                'start_station_id': 'short_name', 
                                'start_station_name': 'name',
                                'start_lat': 'latitude', 'start_lng': 'longitude'
                })
end_stns = end_stn_info.rename(columns = {'end_station_id':'short_name' , 
                                          'end_station_name':'name', 
                                          'end_lat':'latitude', 
                                          'end_lng':'longitude'
                                         })
start_stns.shape,end_stns.shape


In [ ]:
#concat the two files with missing stations
master_missing_stns = pd.concat([start_stns,end_stns]).drop_duplicates(subset='short_name')
master_missing_stns.shape


- Add the station_id, capacity and is_active columns

In [ ]:

#add station_id
master_missing_stns['station_id'] = 'HIST_'+ master_missing_stns['short_name'].astype(str)
master_missing_stns['is_active'] = True
master_missing_stns['capacity'] = None
master_missing_stns.head()

- Test for any test stations and remove them

In [ ]:
test = master_missing_stns

- Keep only the new stations

In [ ]:
master_missing_stns = master_missing_stns[master_missing_stns['short_name'].astype(str).isin(new_stations)]
master_missing_stns.shape

### Insert January trips

In [ ]:
#connect to db
#Connect to database citikike
try:
    conn = psycopg2.connect(
            host = DB_HOST,
            database = DB_NAME,
            user = DB_USER,
            password = DB_PASSW,
            port = DB_PORT,
            keepalives=1,
            keepalives_idle=30,
            keepalives_interval=10,
            keepalives_count=5
    )
    print('Connection Successful')

    cursor = conn.cursor()
except Exception as e:
    print(f'Could not establish connection:{e}')

In [ ]:
conn.rollback()


In [ ]:
tuple_records = [(row['station_id'], row['name'], row['short_name'], row['latitude'], row['longitude'], row['capacity'], row['is_active']) 
                              for _, row in master_missing_stns.iterrows()]


#insert values
try:
    execute_values(
        cursor,
        """
        INSERT INTO stations (station_id, name, short_name, latitude, longitude, capacity, is_active)
        VALUES %s
        """,
        tuple_records
    )
    conn.commit()
    print(f'Inserted {len(tuple_records)}')
except Exception as e:
    #prevents a failed partial insert from leaving the database in a messy state
    conn.rollback()
    print(f'Error:{e}')

In [ ]:
cursor.execute("select short_name from stations where short_name = '5872.1';")
cursor.fetchone()

### Add the new data to trips

In [ ]:
filelist = [data_dir / '202601-citibike-tripdata' / '202601-citibike-tripdata_1.csv', data_dir / '202601-citibike-tripdata' / '202601-citibike-tripdata_2.csv']

for file in filelist:
    load_trips_data(file,conn)


In [ ]:
cursor.execute('select count(*) from trips;')
cursor.fetchone()

- Varify that rows were inserted

In [ ]:
#print number of rows in table
cursor.execute('SELECT COUNT(*) FROM trips;')
cursor.fetchone()



In [ ]:
#print a row
cursor.execute('SELECT * FROM trips LIMIT 1')
cursor.fetchone()

In [ ]:

# Ensure the connection is closed regardless of errors
if conn is not None:
    print("Closing connection.")
    conn.close()

# Station status Live data

- Download one status file:  
    - type in the termina:`scp -i your-key.pem <username>@<instance-ip>:/path/to/remote/file /path/to/local/destination`

In [ ]:
data_dir / 'status_20260130_001249.json'

In [ ]:

live_data_file = data_dir / 'status_20260130_001249.json'

with open(live_data_file, 'r') as file:
    live_data = json.load(file)

In [ ]:
print(live_data['data']['stations'][2000])

In [ ]:
live_df = pd.DataFrame(live_data['data']['stations'])
live_df.shape

In [ ]:
live_df.columns.tolist()

In [ ]:
live_df.head()


In [ ]:
#explore
conn = None
try:
    conn = psycopg2.connect(
            host = DB_HOST,
            database = DB_NAME,
            user = DB_USER,
            password = DB_PASSW,
            port = DB_PORT
    )
    print('Connection Successful')
    cursor = conn.cursor()
    #cursor.execute("SELECT table_name FROM information_schema.tables WHERE table_type = 'BASE TABLE';");
    cursor.execute('''
        SELECT constraint_name 
        FROM information_schema.table_constraints 
        WHERE table_name = 'availability_snapshots' 
            AND constraint_type ='FOREIGN KEY'; ''')
    print(cursor.fetchall())
    
except Exception as e:
    print(f'Could not establish connection:{e}')

finally:
    if conn is not None:
        conn.close()
        print('Connection is closed')

In [ ]:
conn = None
try:
    conn = psycopg2.connect(
            host = DB_HOST,
            database = DB_NAME,
            user = DB_USER,
            password = DB_PASSW,
            port = DB_PORT,
            keepalives=1,
            keepalives_idle=30,
            keepalives_interval=10,
            keepalives_count=5
    )
    cursor = conn.cursor()
    print('Connection Successful')
    
    
except Exception as e:
    print(f'Could not establish connection:{e}')

### - Remove foreign key from table snapshots
- some tables might not match to stations table because some might be removed or added without FK the data is loaded 

In [ ]:

#cursor.execute("SELECT table_name FROM information_schema.tables WHERE table_type = 'BASE TABLE';");
try:
    cursor.execute('''
        ALTER TABLE availability_snapshots 
        DROP CONSTRAINT "availability_snapshots_Station_id_fkey";
        ''')
    conn.commit()
    print('Commited')
except Exception as e:
    print(f'Error: {e}')
    conn.rollback()

In [ ]:
cursor.execute('''
    SELECT constraint_name 
    FROM information_schema.table_constraints 
    WHERE table_name = 'availability_snapshots' 
        AND constraint_type ='FOREIGN KEY'; ''')

print(cursor.fetchall())

- Change the column name Station_id (uppercase) to station_id (lowercase)

In [ ]:
cursor.execute('ALTER TABLE availability_snapshots RENAME COLUMN "Station_id" TO station_id;')
conn.commit()

#### Uploaded all the availability_snapshot data to table, checking

In [ ]:
cursor.execute(
    '''
    SELECT 
        COUNT(*) AS totla_rows,
        COUNT(DISTINCT station_id) AS unique_stations,
        MIN(captured_at) AS earliest_record,
        MAX(captured_at) AS latest_record,
        COUNT(*) FILTER (WHERE num_bikes_available IS NULL) AS null_bikes,
        COUNT(*) FILTER (WHERE num_bikes_available <0) AS neg_bikes
    FROM availability_snapshots;
    
    '''
    )
cursor.fetchone()

In [ ]:
if conn is not None:
    conn.close()
    print('Connection is closed')

In [ ]:
live_df['is_installed'].value_counts()

In [ ]:
live_df['vehicle_types_available'][0]

# Compare the three tables if they match on station_id which is going to connect all the tables

In [ ]:
#station and live data
stn_live = sn_info_df['station_id'] == live_df['station_id']
stn_live.sum()

In [ ]:
print(f'Station info keys: {sn_info.keys()}, keys within data:{sn_info["data"].keys()}, \n\
live data keys: {live_dt.keys()}, keys within data:{live_dt["data"].keys()}')

## Get station ids from both data

In [ ]:
info_stn_ids = [s['station_id'] for s in sn_info['data']['stations']]
live_stn_ids = [s['station_id']for s in live_dt['data']['stations']]

In [ ]:
print(f'info stn id count = {len(info_stn_ids)}')
print(f'live stn id count = {len(live_stn_ids)}')

### print first 5

In [ ]:
print(f'sample station ids from info data:{info_stn_ids[:5]}')
print(f'sample station ids from status data: {live_stn_ids[:5]}')

### Print their types

In [ ]:
print(f'type station ids from info data:{type(info_stn_ids[0])}')
print(f'type station ids from status data: {type(live_stn_ids[0])}')

Do they match

In [ ]:
info_set = set(info_stn_ids)

live_set = set(live_stn_ids)
len(info_set),len(live_set)
match = info_set & live_set
not_match = info_set - live_set
print(f'{len(match)} match and {len(not_match)} does not match')

### Check if the csv file station_id matches with the station info station_id

In [ ]:
start_stn_id_set = set(historic_dt['start_station_id'])
end_stn_id_set = set(historic_dt['end_station_id'])
all_ids = start_stn_id_set | end_stn_id_set
len(all_ids)

In [ ]:
all_ids & info_set
#nothing matches

### Explore names

In [ ]:
sn_info_df.head(2)

In [ ]:
historic_dt.head(2)

In [ ]:
stn_names_info = {s['name'] for s in sn_info['data']['stations']}
trip_names_start = set(historic_dt['start_station_name'].dropna().astype(str))
trip_names_end = set(historic_dt['end_station_name'].dropna().astype(str))
len(trip_names_end)

In [ ]:
print(f'Number of stations in csv start stations:{len(trip_names_start)} and in info data:{len(stn_names_info)}')
print(f'The number of matches for start stations with info= {len(trip_names_start & stn_names_info)}')
print(f'Number of stations in csv end stations:{len(trip_names_end)}')
print(f'The number of matches for end stations with info= {len(trip_names_end & stn_names_info)}')

- lat long

In [ ]:
stn_lat_info = {s['lat'] for s in sn_info['data']['stations']}
trip_lat_start = set(historic_dt['start_lat'].dropna())
trip_lat_end = set(historic_dt['end_lat'].dropna())
len(trip_lat_end)

In [ ]:
print(f'Number of stations in csv start stations:{len(trip_lat_start)} and in info data: {len(stn_lat_info)}')

print(f'The number of matches for start stations with info= {len(trip_lat_start & stn_lat_info)}')
print(f'Number of stations in csv end stations:{len(trip_names_end)}')
print(f'The number of matches for end stations with info= {len(trip_names_end & stn_names_info)}')

In [ ]:
trip_names_end - stn_names_info